In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt

from RRAM import Percolation, CurentSolver, ElectricField, Generation, Temperature
from tqdm import tqdm

import pandas as pd

# Creo el excel donde voy a sacar todos los datos
df = pd.DataFrame(columns=['Tiempo simulacion', 'Voltaje', 'Campo Eléctrico', 'Corriente', 'Temperatura',
                           'Probabilidad de Recombinación', 'Coeficiente difusion'])

In [2]:
import math
import numpy as np
from icecream import ic

from scipy.constants import elementary_charge
from RRAM.Constants import t_0, E_a, k_b_ev, beta_0, L_p, E_m, gamma_drift, DifussiveBehaviour


def recombination_prueba(simulation_time: float, pos_x: int, electric_field: float,
                         temperature: float, grid_size=0.25e-9):

    Oxigen_Ion_velocity = t_0 * grid_size * \
        math.exp(-E_m / (k_b_ev * temperature)) * \
        math.sinh((2*elementary_charge * electric_field * gamma_drift) / (k_b_ev * temperature))

    Prob_in_equilibrio = (simulation_time * t_0) * (math.exp(-E_a / (k_b_ev * temperature)))

    Dif_coef = DifussiveBehaviour(pos_x, Oxigen_Ion_velocity, simulation_time, grid_size)

    Recombination_Probability = beta_0 * (math.exp(-(simulation_time * Oxigen_Ion_velocity) / L_p)) * Dif_coef

    Generation_prob = Prob_in_equilibrio * Recombination_Probability

    # ic(Oxigen_Ion_velocity, Prob_in_equilibrio, Recombination_Probability, Generation_prob, Dif_coef)

    return (Generation_prob, Dif_coef)

In [3]:
espesor_dispositivo = 10            # nm
Atom_size = 0.25                    # nm

eje_x = round(espesor_dispositivo / Atom_size)
eje_y = round(espesor_dispositivo / Atom_size)

num_trampas = 1

# FIXME: Hay una zona donde nunca se ponen trampas
actual_state = Generation.initial_state(eje_x, eje_y, num_trampas)


total_simulation_time = 1
num_pasos = 1000
paso_temporal = total_simulation_time / num_pasos

voltaje_final = 3

paso_guardar = 10

configuraciones_matriz = np.zeros((int((num_pasos / paso_guardar)), eje_x, eje_y))

# Configuraciones iniciales:
Temperatura = 350
Campo_Electrico = 0
voltaje = 0
simulation_time = 0
Corriente = 0

In [4]:
for k in tqdm(range(0, num_pasos)):
    # Guardo el estado anterior
    last_state = actual_state

    simulation_time = paso_temporal*k

    # Calculo la corriente
    voltaje += voltaje_final * paso_temporal

    # Obtengo la corrriente, antes decido cual usar comprobando si ha percolado o no
    # TODO: Revisar por qué me dice que ha percolado si no lo ha hecho
    if Percolation.is_path(actual_state):
        # Si ha percolado uso la corriente de percolación
        # Corriente = CurentSolver.OmhCurrent(Temperatura, Campo_Electrico)
        print("Ha percolado")
    else:
        # Si no ha percolado uso la corriente de campo
        # TODO: REVISAR QUE LA CORRIENTE TIENE LAS UNIDADES CORRECTAS PORQUE NO CUADRAN VALORES.
        Corriente = 100000*CurentSolver.poole_frenkel(Temperatura, Campo_Electrico)

    # Obtengo los valores del campo eléctrico y la temperatura
    Campo_Electrico = ElectricField.SimpleElectricField(voltaje, espesor_dispositivo*1e-9)
    Temperatura = Temperature.Temperature_Joule(voltaje, Corriente, T_0=350)

    # Calculo la probabilidad de generación o recombinación para ello recorro toda la matriz
    for i in range(eje_x):
        for j in range(eje_y):
            if actual_state[i, j] == 0:
                # TODO: REVISAR PROBABILIDAD QUE A VECES SALE MAYOR DE 1
                # TODO: HACER UN REESCALADO DE LOS VALORES PARA EVITAR TENER QUE TRABAJAR CON NUMEROS TAN GRANDES
                prob_generacion = Generation.generation(paso_temporal, Campo_Electrico, Temperatura)
                if (i == 0 and j == 0):
                    # genero un número aleatorio entre 0 y 1
                    random_number = np.random.rand()
                if random_number < prob_generacion:
                    actual_state[i, j] = 1  # Generación
            else:
                # TODO: REVISAR PROBABILIDAD QUE A VECES SALE MAYOR DE 1
                # TODO: HACER UN REESCALADO DE LOS VALORES PARA EVITAR TENER QUE TRABAJAR CON NUMEROS TAN GRANDES
                prob_recombinacion, Dif_coef = recombination_prueba(paso_temporal, i, Campo_Electrico, Temperatura)

                # genero un número aleatorio entre 0 y 1
                random_number = np.random.rand()
                if random_number < prob_recombinacion:
                    actual_state[i, j] = 0  # Recombinación

    # Guardo los datos en el excel
    df.loc[k] = [simulation_time, voltaje, Campo_Electrico, Corriente, Temperatura, prob_recombinacion, Dif_coef]

    # Guardo el estado actual CADA paso_guardar PASOS MONTECARLO
    if k % paso_guardar == 0:
        configuraciones_matriz[int(k / paso_guardar) - 1] = actual_state

# Guardo los datos en un excel
df.to_excel('resultados_pruebas.xlsx', index=False)

  0%|          | 0/1000 [00:00<?, ?it/s]

 84%|████████▍ | 844/1000 [00:07<00:01, 120.30it/s]

Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado


 86%|████████▌ | 857/1000 [00:07<00:02, 58.51it/s] 

Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado


...

Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado


100%|█████████▉| 997/1000 [00:12<00:00, 26.34it/s]

Ha percolado
Ha percolado
Ha percolado
Ha percolado
Ha percolado


100%|██████████| 1000/1000 [00:13<00:00, 76.54it/s]


Ha percolado
